# LeetCode #1242: Web Crawler Multithreaded

https://leetcode.com/problems/web-crawler-multithreaded/

## Synchronization Approaches

| Approach | Mechanism | Notes |
| :--- | :--- | :--- |
| **Naive: Single-Threaded BFS** | Sequential queue | Correct but slow; blocks on each HTTP call |
| **Optimal: Concurrent BFS with Thread Pool ★** | `ConcurrentQueue` + `Task`/`Thread` pool | Multiple URLs fetched in parallel; visited set protected by lock |

---

## Understanding the Methods

### Naive: Single-Threaded BFS
Standard BFS: dequeue a URL, call `getUrls`, enqueue new same-host URLs, track visited. Correct but serial — each slow HTTP call blocks progress.

### Optimal: Concurrent BFS with Thread Pool ★
Use a thread-safe visited set (guarded by `lock`) and a work queue. Spawn a task for each new URL; each task calls `getUrls`, filters new same-host URLs, and enqueues further work. A countdown mechanism (e.g., `CountdownEvent`) signals when all outstanding tasks finish.

**Constraints:**
* `1 <= urls.length <= 1000`
* Only crawl URLs with the same hostname as `startUrl`
* `HtmlParser.getUrls(url)` returns all hyperlinks on the page


## Solutions
### C#

In [ ]:
using System;
using System.Collections.Concurrent;
using System.Collections.Generic;
using System.Linq;
using System.Threading;
using System.Threading.Tasks;

public class Solution
{
    public IList<string> Crawl(string startUrl, HtmlParser htmlParser)
    {
        string host = GetHost(startUrl);
        var visited = new ConcurrentDictionary<string, bool>();
        visited[startUrl] = true;

        var pending = new ConcurrentQueue<Task>();
        var done    = new CountdownEvent(1);

        void Crawl(string url)
        {
            foreach (var next in htmlParser.GetUrls(url))
            {
                if (GetHost(next) == host && visited.TryAdd(next, true))
                {
                    done.AddCount();
                    pending.Enqueue(Task.Run(() => { Crawl(next); done.Signal(); }));
                }
            }
        }

        Task.Run(() => { Crawl(startUrl); done.Signal(); });
        done.Wait();
        return visited.Keys.ToList();
    }

    private static string GetHost(string url)
    {
        // "http://hostname/path" -> "hostname"
        int start = url.IndexOf("//") + 2;
        int end   = url.IndexOf('/', start);
        return end == -1 ? url[start..] : url[start..end];
    }
}

### Python

In [ ]:
from typing import List
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse
import threading

class Solution:
    def crawl(self, startUrl: str, htmlParser: 'HtmlParser') -> List[str]:
        host = urlparse(startUrl).netloc
        visited = {startUrl}
        lock = threading.Lock()
        queue = deque([startUrl])

        def fetch(url: str) -> List[str]:
            return [
                u for u in htmlParser.getUrls(url)
                if urlparse(u).netloc == host
            ]

        with ThreadPoolExecutor() as pool:
            futures = {pool.submit(fetch, startUrl)}
            while futures:
                done_futures = set()
                for f in as_completed(futures):
                    done_futures.add(f)
                    for url in f.result():
                        with lock:
                            if url not in visited:
                                visited.add(url)
                                futures.add(pool.submit(fetch, url))
                futures -= done_futures

        return list(visited)

### Go

In [ ]:
package main

import (
	"sync"
	"strings"
)

func crawl(startUrl string, htmlParser HtmlParser) []string {
	host := getHost(startUrl)
	var mu sync.Mutex
	visited := map[string]bool{startUrl: true}
	var wg sync.WaitGroup

	var visit func(url string)
	visit = func(url string) {
		defer wg.Done()
		for _, next := range htmlParser.GetUrls(url) {
			if getHost(next) != host { continue }
			mu.Lock()
			new_ := !visited[next]
			if new_ { visited[next] = true }
			mu.Unlock()
			if new_ { wg.Add(1); go visit(next) }
		}
	}

	wg.Add(1)
	go visit(startUrl)
	wg.Wait()

	mu.Lock()
	defer mu.Unlock()
	result := make([]string, 0, len(visited))
	for u := range visited { result = append(result, u) }
	return result
}

func getHost(url string) string {
	// "http://host/path" -> "host"
	s := strings.TrimPrefix(url, "http://")
	if i := strings.Index(s, "/"); i != -1 { return s[:i] }
	return s
}

### Rust

In [ ]:
use std::collections::HashSet;
use std::sync::{Arc, Mutex};
use std::thread;

fn crawl(start_url: String, html_parser: Arc<dyn HtmlParser + Send + Sync>) -> Vec<String> {
    let host = get_host(&start_url);
    let visited = Arc::new(Mutex::new(HashSet::from([start_url.clone()])));
    let mut handles = vec![];

    let pending = Arc::new(Mutex::new(vec![start_url]));

    loop {
        let batch: Vec<String> = std::mem::take(&mut *pending.lock().unwrap());
        if batch.is_empty() { break; }

        let mut new_handles = vec![];
        for url in batch {
            let host = host.clone();
            let visited = Arc::clone(&visited);
            let pending = Arc::clone(&pending);
            let parser  = Arc::clone(&html_parser);
            new_handles.push(thread::spawn(move || {
                for next in parser.get_urls(&url) {
                    if get_host(&next) == host {
                        let mut v = visited.lock().unwrap();
                        if v.insert(next.clone()) {
                            pending.lock().unwrap().push(next);
                        }
                    }
                }
            }));
        }
        for h in new_handles { h.join().unwrap(); }
    }

    Arc::try_unwrap(visited).unwrap().into_inner().unwrap().into_iter().collect()
}

fn get_host(url: &str) -> String {
    let s = url.trim_start_matches("http://");
    s.split('/').next().unwrap_or("").to_string()
}

## Concurrency Scenarios

1. **Single page with no links**: Only `startUrl` is visited; no tasks spawned; `CountdownEvent` fires immediately after the first task completes.
2. **Cross-host links ignored**: Fetching page A returns URLs for host B; `GetHost` check filters them out — they never enter `visited`.
3. **Duplicate URL race**: Two threads fetch pages that both link to the same new URL; `TryAdd` / `inserted` ensures only one task is spawned for it.
4. **Deep link graph (chain A→B→C→D)**: Each level spawns one task; `CountdownEvent` incremented before dispatch and decremented on completion tracks the wavefront correctly.
5. **Wide graph (star from start)**: 999 links from `startUrl` all enqueued as parallel tasks; thread pool limits concurrency while `CountdownEvent` tracks all 999 completions.
